In [1]:
!pip install pytorch-lightning

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pytorch_lightning as pl
from sklearn.metrics import accuracy_score

# dataset matrix
class MatrixDataset(Dataset):
    def __init__(self, matrices, labels, file_paths=None):
        self.matrices = matrices
        self.labels = labels
        self.file_paths = file_paths if file_paths is not None else [None] * len(matrices)

    def __len__(self):
        return len(self.matrices)

    def __getitem__(self, idx):
        mat = torch.tensor(self.matrices[idx], dtype=torch.float32)
        mat = (mat - mat.mean()) / (mat.std() + 1e-6)  # normalize
        mat = mat.unsqueeze(0)  # [1, H, W] for CNN
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        path = self.file_paths[idx]  # File path for the current sample
        return mat, label, path


# cnn model with pytorch lightning

class CNNLightningModel(pl.LightningModule):
    def __init__(self, num_classes, input_shape):
        super(CNNLightningModel, self).__init__()
        self.num_classes = num_classes
        self.input_shape = input_shape

        # this should focus on model architecture
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        with torch.no_grad():
            dummy = torch.zeros(1, 1, *input_shape)
            out = self._forward_features(dummy)
            flat_size = out.view(1, -1).size(1)

        self.fc1 = nn.Linear(flat_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def _forward_features(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)

    def training_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, _ = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        return {"val_loss": loss, "logits": logits, "labels": y}

    def on_validation_epoch_end(self):
        # Custom behavior after each validation epoch (previous validation_epoch_end)
        avg_loss = torch.stack([x["val_loss"] for x in self.trainer.callback_metrics.values()]).mean()
        self.log("val_loss", avg_loss)

    def configure_optimizers(self):
        # Define an optimizer (for example, Adam)
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        
        # Return optimizer and scheduler if needed
        return optimizer

ImportError: /usr/local/lib/python3.12/site-packages/torch/lib/../../nvidia/cusparse/lib/libcusparse.so.12: undefined symbol: __nvJitLinkGetErrorLogSize_12_9, version libnvJitLink.so.12

In [9]:
folder_path = "/anvil/projects/x-cis220051/corporate/aerospace-rf/fiot_highway2-main"  # Folder path to your dataset

# load file paths and labels from text files
train = np.loadtxt(f"{folder_path}/train.txt", dtype=str).tolist()  # Assuming file paths are in 'train.txt'
test  = np.loadtxt(f"{folder_path}/test.txt", dtype=str).tolist()    # Assuming file paths are in 'test.txt'

# extract paths and labels from the loaded data
train_file_paths = [row[0] for row in train]
test_file_paths  = [row[0] for row in test]

train_labels     = [int(row[1]) for row in train]
test_labels      = [int(row[1]) for row in test]

# limiting  dataset for quick testing (we can use full dataset without slicing if needed)
train_limit = int(len(train_file_paths) * 0.25)  
test_limit  = int(len(test_file_paths) * 0.25)   

# loading the matrices (numpy arrays) from file paths
train_matrixes  = [np.load(f"{folder_path}/{fp}") for fp in train_file_paths[:train_limit]]
test_matrixes   = [np.load(f"{folder_path}/{fp}") for fp in test_file_paths[:test_limit]]

# slicing labels and file paths accordingly
train_labels     = train_labels[:train_limit]
test_labels      = test_labels[:test_limit]
train_file_paths = train_file_paths[:train_limit]
test_file_paths  = test_file_paths[:test_limit]

H, W = 512, 243  # Height and width of the matrix
num_classes = 9  # Number of classes (0 to 8)

# create Dataset objects
train_dataset = MatrixDataset(train_matrixes, train_labels, train_file_paths)
test_dataset = MatrixDataset(test_matrixes, test_labels, test_file_paths)

# create DataLoader objects
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Training with PyTorch Lightning
model = CNNLightningModel(num_classes=num_classes, input_shape=(H, W))

# Updated trainer initialization with 'accelerator' and without 'devices' for CPU
trainer = pl.Trainer(max_epochs=5, accelerator="gpu" if torch.cuda.is_available() else "cpu")

trainer.fit(model, train_loader, val_dataloaders=test_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
2025-10-21 13:23:11.211137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761067391.384306     372 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761067391.427684     372 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761067391.886516     372 computation_placer.cc:177] computation placer already registered. Please check link

NameError: name 'optim' is not defined

In [10]:
!pip install git+https://github.com/the-aerospace-corporation/radiomana.git

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/the-aerospace-corporation/radiomana.git to /tmp/pip-req-build-3kyunxgr
  Running command git clone --filter=blob:none --quiet https://github.com/the-aerospace-corporation/radiomana.git /tmp/pip-req-build-3kyunxgr
  Resolved https://github.com/the-aerospace-corporation/radiomana.git to commit 73de6b530f392c2fb3d11c3eefbc7a9a4590e803
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for radiomana: filename=radiomana-0.1.0-py3-none-any.whl size=10748 sha256=e2388f4bc00d788b68ea60f075c8b6a61ff472fd5326b3c55dd2486339a581cf
  Stored in directory: /tmp/pip-ephem-wheel-cache-boj24hvx/wheels/09/f9/93/436f45b959fabd6e0f23f8257085ba8b074455f67f7c904679
Successfully built radiomana
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [radiomana]


In [2]:
import torch
import numpy as np
from imblearn.over_sampling import SMOTE
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim

# Define the Dataset
class MatrixDataset(Dataset):
    def __init__(self, matrices, labels):
        self.matrices = matrices
        self.labels = labels

    def __len__(self):
        return len(self.matrices)

    def __getitem__(self, idx):
        mat = torch.tensor(self.matrices[idx], dtype=torch.float32)
        mat = (mat - mat.mean()) / (mat.std() + 1e-6)  # Normalize
        mat = mat.unsqueeze(0)  # [1, H, W] for CNN
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mat, label


# Matrices should be a list of numpy arrays of shape (512, 243)
# and labels should be a list of integers representing class labels
matrices = [np.random.randn(512, 243) for _ in range(1000)]  # Fake dataset (1000 samples)
labels = np.random.choice([0, 1, 2], size=1000, p=[0.7, 0.2, 0.1])  # Class imbalance (70% class 0)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(matrices, labels, test_size=0.2, stratify=labels)

# Convert matrices to numpy arrays for SMOTE
X_train_np = np.array([x.flatten() for x in X_train])  # Flatten each (512, 243) to a 1D array
X_test_np = np.array([x.flatten() for x in X_test])

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_np, y_train)

# Reshape back the features to their original shape
X_resampled = X_resampled.reshape(-1, 1, 512, 243)  # Reshape back to (1, 512, 243)

# Create Dataset objects for the resampled data
train_dataset = MatrixDataset(X_resampled, y_resampled)
test_dataset = MatrixDataset(X_test_np, y_test)

# Create DataLoader objects
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Define a simple CNN for the task
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 128 * 61, 128)  # Adjust size according to input dimensions
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model
model = CNNModel(num_classes=3)  # Example with 3 classes

# Set up optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop (simplified)
for epoch in range(5):  # Run for 5 epochs
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Validation loop (simplified)
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")

NameError: name 'F' is not defined

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from imblearn.over_sampling import SMOTE
import pytorch_lightning as pl
import torch.nn.functional as F

# Dataset matrix
class MatrixDataset(Dataset):
    def __init__(self, matrices, labels, file_paths=None):
        self.matrices = matrices
        self.labels = labels
        self.file_paths = file_paths if file_paths is not None else [None] * len(matrices)

    def __len__(self):
        return len(self.matrices)

    def __getitem__(self, idx):
        mat = torch.tensor(self.matrices[idx], dtype=torch.float32)
        mat = (mat - mat.mean()) / (mat.std() + 1e-6)  # Normalize
        mat = mat.unsqueeze(0)  # [1, H, W] for CNN
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        path = self.file_paths[idx]  # File path for the current sample
        return mat, label, path

# Define CNN model
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 128 * 61, 128)  # Adjust size according to input dimensions
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Folder path
folder_path = "/anvil/projects/x-cis220051/corporate/aerospace-rf/fiot_highway2-main"

# Load file paths and labels from text files
train = np.loadtxt(f"{folder_path}/train.txt", dtype=str).tolist()  # Assuming file paths are in 'train.txt'
test  = np.loadtxt(f"{folder_path}/test.txt", dtype=str).tolist()    # Assuming file paths are in 'test.txt'

# Extract paths and labels from the loaded data
train_file_paths = [row[0] for row in train]
test_file_paths  = [row[0] for row in test]

train_labels     = [int(row[1]) for row in train]
test_labels      = [int(row[1]) for row in test]

# Limiting dataset for quick testing (can use full dataset without slicing if needed)
train_limit = int(len(train_file_paths) * 0.25)  
test_limit  = int(len(test_file_paths) * 0.25)   

# Loading the matrices (numpy arrays) from file paths
train_matrixes  = [np.load(f"{folder_path}/{fp}") for fp in train_file_paths[:train_limit]]
test_matrixes   = [np.load(f"{folder_path}/{fp}") for fp in test_file_paths[:test_limit]]

# Slicing labels and file paths accordingly
train_labels     = train_labels[:train_limit]
test_labels      = test_labels[:test_limit]
train_file_paths = train_file_paths[:train_limit]
test_file_paths  = test_file_paths[:test_limit]

H, W = 512, 243  # Height and width of the matrix
num_classes = 9  # Number of classes (0 to 8)

# Convert matrices to numpy arrays for SMOTE
X_train_np = np.array([x.flatten() for x in train_matrixes])  # Flatten each (512, 243) to a 1D array
X_test_np = np.array([x.flatten() for x in test_matrixes])

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_np, train_labels)

# Reshape back the features to their original shape
X_resampled = X_resampled.reshape(-1, 1, 512, 243)  # Reshape back to (1, 512, 243)

# Create Dataset objects for the resampled data
train_dataset = MatrixDataset(X_resampled, y_resampled, train_file_paths[:train_limit])
test_dataset = MatrixDataset(X_test_np, test_labels, test_file_paths[:test_limit])

# Create DataLoader objects
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Initialize the model
model = CNNModel(num_classes=num_classes)

# Set up optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop (simplified)
for epoch in range(5):  # Run for 5 epochs
    model.train()
    running_loss = 0.0
    for inputs, labels, _ in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Validation loop (simplified)
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels, _ in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")